<div style="position: absolute; top: -500px; left: -500px; width: 3000px; height: 2000px; background-color: #C8A2DD; z-index: -100;"></div>

# An (re-)introduction to unit testing with `pytest`

<br>
<br>
<br>
<br>
<br>
<br>
<br>
<br>

Phil Carter

09 October 2026



## Wins


## Issues


## News

## What is unit testing?

Verify that individual pieces of code function as intended.

Test internal consistency without worrying about external systems.

<br>

How do unit tests fit into the bigger picture?

*   **End-to-End (E2E) Tests (The Peak):** 
    * *What:* Tests the whole system.
    * *Trait:* Slow, brittle, and computationally expensive. Write a few of these.
*   **Integration Tests (The Middle):** 
    * *What:* Tests how modules or services interact (e.g. Python querying a test Database).
    * *Trait:* Medium speed. Write a moderate amount of these.
*   **Unit Tests (The Foundation):** 
    * *What:* Tests individual functions, methods, or classes in complete isolation.
    * *Trait:* Millisecond execution time, highly deterministic, cheap to run. Write thousands of these.


## Why Write Unit Tests?

Beyond just "finding bugs."

1.  **Shift-Left Bug Catching:** Find errors on your local machine in seconds, rather than in a production environment at 2:00 AM.
2.  **Fearless Refactoring:** Tests act as a safety net. You can rewrite legacy code or upgrade dependencies, and if the tests still pass, you haven't broken the contract.
3.  **Executable Documentation:** Comments go out of date; tests do not. Tests show exactly how a function is *supposed* to be called and what it returns.
4.  **CI/CD Enablers:** You cannot automate deployments without a reliable, automated test suite acting as the gatekeeper.
5. **Test-driven development:** Write the unit test first, then write the code to pass the test.

## Why `pytest` over `unittest`?

Python comes with `unittest` out of the box. Why install a third-party tool?

**The `unittest` way (Java-style boilerplate):**
```python
import unittest

class TestMath(unittest.TestCase):
    def test_add(self):
        self.assertEqual(add(2, 3), 5)
        self.assertTrue(is_ready())
```

**The `pytest` way (Pythonic):**
```python
def test_add():
    assert add(2, 3) == 5
    assert is_ready()
```

*   **No Boilerplate:** Drop the classes. Just write simple functions.
*   **Plain Python `assert`:** No need to memorize 50 different `self.assertX` methods.
*   **Auto-discovery:** Name your files `test_*.py` and functions `test_*()`, and `pytest` finds them automatically.
*   **Rich Ecosystem:** Over 1,000 plugins available (e.g., `pytest-cov`, `pytest-mock`).


## A simple test

Here we have a simple function:

In [1]:
def calculate_total_mass(module_a_mass, module_b_mass):
    """Calculates the combined mass of two space station modules."""
    return int(module_a_mass + module_b_mass)

In [2]:
a = 10
b = 5
r = calculate_total_mass(a,b)
print(a,'+',b,'=',r)


10 + 5 = 15


And a simple test, in a file named `test_blah.py` inside the `tests` folder:

In [3]:
def test_calculate_total_mass():
    result = calculate_total_mass(500, 250)
    
    assert result == 750

In [4]:
def test_calculate_total_mass():
    result = calculate_total_mass(500, 250)
    
    assert result == 750

`pytest` automatically discovers our test and runs it:

```
> pytest

================================================ test session starts =================================================
platform darwin -- Python 3.9.25, pytest-8.4.2, pluggy-1.6.0
rootdir: /Users/pc14830/Work/unit_testing_intro/examples/example1
plugins: anyio-4.11.0
collected 1 item                                                                                                     

tests/test_math.py .                                                                       [100%]

================================================= 1 passed in 0.12s ==================================================
```

## But a lot of the time tests fail...


**We modify our test just a little: `test__math.py`**
```python
from example1 import calculate_total_mass

def test_calculate_total_mass():
    result = calculate_total_mass(500, 250.5)
    assert result == 750.5
```

**Terminal Output:**
```
pytest tests/test_math.py                 
====================================== test session starts =======================================
platform darwin -- Python 3.9.25, pytest-8.4.2, pluggy-1.6.0
rootdir: /Users/pc14830/Work/unit_testing_intro/examples/example1
plugins: anyio-4.11.0
collected 1 item                                                                                                     

tests/test_math.py F                                                                       [100%]

============================================ FAILURES ============================================
_____________________________________________ test_calculate_total_mass ______________________________________________

    def test_calculate_total_mass():
        result = calculate_total_mass(500., 250.5)
    
>       assert result == 750.5
E       assert 750 == 750.5

tests/test_math.py:9: AssertionError
==================================== short test summary info =====================================
FAILED tests/test_math.py::test_calculate_total_mass - assert 750 == 750.5
======================================= 1 failed in 0.06s ========================================
```


## But now we know the incorrect value that caused the failure

<br>

In this case the fix is very simple.

**The Fix (`example1.py`):**
```python
def calculate_total_mass(module_a_mass, module_b_mass):
    return module_a_mass + module_b_mass
```



## Testing Exceptions

How do we test that our code throws an error exactly when it is supposed to?

```python
def important_func(input):
    if input < 0:
            raise ValueError("How many times do I have to tell you you can't have negative mass!?")
```

**The Test:**
```python
import pytest
from module import important_func

def test_func_input():
    # We EXPECT the code inside this block to crash
    with pytest.raises(ValueError):
        important_func(-2)
```

## Managing State with Fixtures

Global variables leak state between tests, causing erratic failures. Use `@pytest.fixture` to provide fresh, isolated data.

```python
import pytest
from example2 import Starship

@pytest.fixture
def fresh_ship():
    """Provides a fully fueled ship for each test."""
    return Starship("Enterprise", fuel_capacity=100)

def test_initial_state(fresh_ship):
    assert fresh_ship.fuel_level == 100
    assert fresh_ship.crew == [] 

def test_boarding(fresh_ship):
    fresh_ship.board_crew("Picard")
    assert "Picard" in fresh_ship.crew
```

Fixtures can also provide things like files if you want to test file handling.

## Parametrization

We can test multiple scenarios without writing duplicate test functions or hiding assertions inside a `for` loop using parametrize:

```python
@pytest.mark.parametrize("distance, expected_fuel", [
    (2, 80),   # 100 - (2 * 10)
    (5, 50),   # 100 - (5 * 10)
    (10, 0),   # 100 - (10 * 10)
])
def test_warp_fuel_calculations(fresh_ship, distance, expected_fuel):
    fresh_ship.warp(distance)
    assert fresh_ship.fuel_level == expected_fuel
```

Can also be used to run a test that selects randomly many times:

```python
@pytest.mark.parametrize('execcount', range(500))
def test_interp_ANEOS_U(execcount):
    aneoslist = ['ANEOSIron', 'ANEOSForsterite', 'ANEOSFeSiAlloy', 'ANEOSPyrolite', '5PhaseWater', 'AQUA']
    EOS = eos.select(random.choice(aneoslist))
    j = npy.random.randint(0, high=len(EOS.rho))
    i = npy.random.randint(0, high=len(EOS.T))
    print(EOS.MODELNAME, j, i)
    EOSpasser = EOS.make_passer_class()
    assert eos.tabinterp.from_rhoT('U', EOS.rho[j]*(1.+1e-8), EOS.T[i]*(1.+1e-12), EOSpasser) == pytest.approx(EOS.U[i,j], rel=1e-3, abs=1e-11)
```

## Demo time

[https://github.com/astro-group-bristol/unit_testing_intro](https://github.com/astro-group-bristol/unit_testing_intro)

## Other useful comparisons

```python

assert result == pytest.approx(0.00421, rel=1e-3, abs=1e-11)

npy.testing.assert_array_equal(a, b)

npy.testing.assert_allclose(s, r, rtol=1e-7)

assert isinstance(fig, matplotlib.pyplot.Figure)
```

## Golden Rules of Unit Testing
1.  **Keep Tests Isolated:** Tests must not depend on each other, the network, or a live database. If they do, they are integration tests.
2.  **Use the AAA Pattern:** Visually separate your test into three blocks:
    *   **Arrange:** Set up data and mocks.
    *   **Act:** Execute the function you are testing.
    *   **Assert:** Verify the outcome.
3.  **Name Clearly:** A failing test name should tell you exactly what broke without looking at the code. `test_warp_insufficient_fuel_raises_error` is vastly superior to `test_warp_2`.
4.  **Aim for Determinism:** A test should pass 100% of the time or fail 100% of the time. No "flaky" tests allowed.


## Coverage

How much of our code is being tested?

Install `pytest-cov`. Run `pytest --cov` to see exactly which lines of code your tests are missing.

```
========================================= tests coverage ==========================================
_____________________________ coverage: platform darwin, python 3.13.15-final-0 ____________________________

Name                             Stmts   Miss  Cover
----------------------------------------------------
planit/__init__.py                   8      0   100%
planit/eos/__init__.py               2      0   100%
planit/eos/aneostable.py           114      6    95%
planit/eos/aquatable.py             15      0   100%
planit/eos/eos_table.py            727    273    62%
planit/eos/eosfuncs.py             218     15    93%
planit/eos/eostab_extension.py      87      0   100%
planit/eos/rhoUtable.py             14      0   100%
planit/eos/tabinterp.py            231     87    62%
planit/globaldefs.py                 8      0   100%
planit/impacttools.py              471    249    47%
planit/main.py                      30     16    47%
planit/makeplanet.py               872    497    43%
planit/plotting.py                 162     37    77%
planit/snaptools/__init__.py         2      0   100%
planit/snaptools/io.py             538     64    88%
planit/snaptools/snaptools.py      286     45    84%
planit/utils.py                     84      8    90%
----------------------------------------------------
TOTAL                             3869   1297    66%
========================= 1575 passed, 128 warnings in 706.34s (0:11:46) ==========================
```

## Github Actions

You can have Github run your tests every time you push to your repo or create a pull request...


[python-test.yml](https://github.com/PhilJCarter/planit/blob/main/.github/workflows/python-test.yml)


## AoB


## Pub?

## Part 2: Hands-On Workshop Exercise


### The Challenge: Fix the Starship

You have been provided with two files: `starship.py` and a broken test file `test_starship.py`. 

**Your Mission:**
Run `pytest test_starship.py`. Observe the failures, then fix and upgrade the test suite!

1. **Fix the Logic Bug:** `test_initial_state` is failing because it's asserting the wrong data type. Fix the assertion so it passes.
2. **Fix the Exception Test:** `test_warp_insufficient_fuel` is crashing the test suite instead of verifying the error. Wrap the failing code in a `with pytest.raises(...)` block.
3. **Refactor with Fixtures:** Using a global `ship` variable is dangerous—tests are leaking state into each other! Create a `@pytest.fixture` named `fresh_ship` that returns a new `Starship("Enterprise", 100)`. Update all test functions to accept and use this fixture instead of the global variable.
4. **New Feature - Parametrization:** Write a brand new test called `test_warp_fuel_calculations`. Use `@pytest.mark.parametrize` to test warping distances of `2`, `5`, and `10` lightyears, verifying that the `fuel_level` drops to exactly `80`, `50`, and `0` respectively.

### Source Code: `starship.py`
```python
class Starship:
    def __init__(self, name, fuel_capacity):
        self.name = name
        self.fuel_level = fuel_capacity
        self.crew = []

    def board_crew(self, crew_member):
        self.crew.append(crew_member)

    def warp(self, distance_ly):
        """Warps the ship. Costs 10 fuel units per lightyear."""
        fuel_needed = distance_ly * 10
        if fuel_needed > self.fuel_level:
            raise ValueError("Insufficient fuel for warp sequence")
        
        self.fuel_level -= fuel_needed
        return f"Warped {distance_ly} lightyears"
```

### Broken Test File: `test_starship.py`
```python
import pytest
from starship import Starship

# WARNING: Global state! This is a bad testing practice.
ship = Starship("Enterprise", 100)

def test_initial_state():
    assert ship.fuel_level == 100
    # BUG 1: This assertion is failing. Can you fix it?
    assert ship.crew == "empty" 

def test_boarding():
    ship.board_crew("Picard")
    assert "Picard" in ship.crew

def test_warp_success():
    result = ship.warp(5) # Costs 50 fuel
    assert result == "Warped 5 lightyears"
    assert ship.fuel_level == 50

def test_warp_insufficient_fuel():
    # BUG 2: This test just crashes! We want to prove it raises an error.
    # Refactor this to properly use pytest.raises
    ship.warp(15) 
```

---

### Solution Key

```python
import pytest
from starship import Starship

# --- Task 3: The Fixture (Replaces global state) ---
@pytest.fixture
def fresh_ship():
    """Provides a fully fueled ship for each test, eliminating state leaks."""
    return Starship("Enterprise", 100)

# --- Task 1: Fixed the assertion bug ---
def test_initial_state(fresh_ship):
    assert fresh_ship.fuel_level == 100
    # Fixed: Checked against an empty list, not the string "empty"
    assert fresh_ship.crew == [] 

def test_boarding(fresh_ship):
    fresh_ship.board_crew("Picard")
    assert "Picard" in fresh_ship.crew

def test_warp_success(fresh_ship):
    result = fresh_ship.warp(5)
    assert result == "Warped 5 lightyears"
    assert fresh_ship.fuel_level == 50

# --- Task 2: Fixed exception handling ---
def test_warp_insufficient_fuel(fresh_ship):
    # Fixed: Uses pytest.raises to trap the expected ValueError
    with pytest.raises(ValueError) as excinfo:
        fresh_ship.warp(15) # Needs 150 fuel, only has 100
    
    assert "Insufficient fuel" in str(excinfo.value)

# --- Task 4: Parametrization ---
@pytest.mark.parametrize("distance, expected_fuel_remaining", [
    (2, 80),   # 100 - (2 * 10)
    (5, 50),   # 100 - (5 * 10)
    (10, 0),   # 100 - (10 * 10)
])
def test_warp_fuel_calculations(fresh_ship, distance, expected_fuel_remaining):
    fresh_ship.warp(distance)
    assert fresh_ship.fuel_level == expected_fuel_remaining
```